# Modelo final — BETO + LoRA

In [ ]:
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

In [ ]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

from betohumor.config import DataConfig, BetoConfig, LoraConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split, HahaDataset
from betohumor.models.lora import build_beto_lora
from betohumor.train import train_model
from betohumor.metrics import get_best_macro_f1_from_history, get_training_history
from betohumor.plots import plot_training_curves

/opt/miniconda3/envs/AP_FINAL/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Mejor configuración (resultado del CV)
Completar con los valores reales que ganaron en `cross_validation_lora.ipynb`.

In [2]:
BEST_R     = 32     
BEST_ALPHA = 64   
BEST_LR    = 1e-4  
BEST_WD    = 0.05 

## 2. Datos: train + val combinados


In [5]:
data_config  = DataConfig(data_path = "../data/raw/haha_2019_train.csv")
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)
df_combined = pd.concat([df_train, df_val]).reset_index(drop=True)

df_train_final, df_val_final = train_test_split(
    df_combined, test_size=0.05,
    stratify=df_combined[data_config.label_col],
    random_state=data_config.seed,
)

print(f'Train final: {len(df_train_final)} | Val : {len(df_val_final)}')

Train: 19197 | Val: 2397 | Test: 2400
Train final: 20514 | Val : 1080


## 3. Entrenar el modelo final

In [7]:
tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)

train_dataset = HahaDataset(df_train_final, tokenizer, data_config)
val_dataset   = HahaDataset(df_val_final,   tokenizer, data_config)

lora_config = LoraConfig(r=BEST_R, lora_alpha=BEST_ALPHA)
model = build_beto_lora(beto_config, lora_config)

trainer = train_model(
    model, train_dataset, val_dataset, beto_config,
    output_dir='results/final_lora',
    seed=data_config.seed,
    learning_rate=BEST_LR,
    weight_decay=BEST_WD,
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 63036.15it/s]
BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECT

trainable params: 1,181,186 || all params: 111,033,604 || trainable%: 1.0638


KeyboardInterrupt: 

## 4. Curva de entrenamiento

In [ ]:
history = get_training_history(trainer)
macro_f1 = get_best_macro_f1_from_history(trainer)

print(f'Macro F1 (early stopping val): {macro_f1:.4f}')
plot_training_curves(history)

## 5. Guardar el modelo final

In [ ]:
trainer.save_model('results/final_lora')
tokenizer.save_pretrained('results/final_lora')
print('Modelo final guardado en results/final_lora')